# Motion GIF Visualization from iSign Pose Model
Generate motion GIFs showing pose sequences predicted by the trained iSign model.

## 1. Import Libraries

In [2]:
import sys
import json
from pathlib import Path
import numpy as np
import torch
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from typing import List, Tuple, Optional
import imageio
import warnings
warnings.filterwarnings('ignore')

# Resolve backend paths
BACKEND_DIR = Path('../').resolve()
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

print(f"Backend path: {BACKEND_DIR}")

Backend path: /mnt/d/KATHIR_KALIDASS/CSLR/application


## 2. Load Model and Configuration

In [3]:
# Load training configuration
ckpt_dir = Path('checkpoints/isign_pose_only_npy')
config_path = ckpt_dir / 'train_config.json'
ckpt_path = ckpt_dir / 'best.pt'

cfg = json.loads(config_path.read_text(encoding='utf-8'))
print(f"✓ Config loaded from: {config_path}")
print(f"  - num_frames: {cfg['num_frames']}")
print(f"  - hidden_dim: {cfg['hidden_dim']}")
print(f"  - pose_only: {cfg['pose_only']}")

✓ Config loaded from: checkpoints/isign_pose_only_npy/train_config.json
  - num_frames: 48
  - hidden_dim: 256
  - pose_only: True


In [4]:
# Import model and utilities
import importlib.util

# Load train_isign as a module dynamically from scripts folder
train_isign_path = Path('scripts/train_isign.py').resolve()
if not train_isign_path.exists():
    print(f"Error: train_isign.py not found at {train_isign_path}")
    print(f"Current directory: {Path.cwd()}")
else:
    spec = importlib.util.spec_from_file_location("train_isign_module", train_isign_path)
    train_isign_module = importlib.util.module_from_spec(spec)
    sys.modules['train_isign_module'] = train_isign_module
    spec.loader.exec_module(train_isign_module)

    # Extract the classes and functions
    ISignCSLRModel = train_isign_module.ISignCSLRModel
    load_checkpoint = train_isign_module.load_checkpoint
    decode_sequences = train_isign_module.decode_sequences
    ids_to_tokens = train_isign_module.ids_to_tokens
    print(f"✓ train_isign module loaded from {train_isign_path}")

from app.data.isign_dataset import build_dataloaders

# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Load test dataloader (pose-only, no RGB frames to avoid scipy issues)
data_dir = Path(cfg['data_dir'])
_, _, test_loader, vocab = build_dataloaders(
    data_dir=str(data_dir),
    batch_size=1,
    num_frames=int(cfg['num_frames']),
    frame_size=tuple(cfg['frame_size']),
    num_workers=0,
    use_poses=True,
    use_frames=False,  # Disable RGB frames (pose-only model)
    max_samples=0,
    pin_memory=False,
    persistent_workers=False,
    preload_n=0,
    use_albumentations=False,  # Disable albumentations
    pose_backend=str(cfg.get('pose_backend', 'auto')),
    pose_lmdb_path=cfg.get('pose_lmdb_path'),
    pose_lmdb_readahead=bool(cfg.get('pose_lmdb_readahead', False)),
)
print(f"✓ Test dataloader loaded | vocab_size={len(vocab)}")

✓ train_isign module loaded from /mnt/d/KATHIR_KALIDASS/CSLR/application/backend/scripts/train_isign.py
Device: cuda


17:47:06 | INFO    | app.data.isign_dataset | [ISignDataset] split=train  samples=101452  vocab=37535  pose_dim=1728  preload_n=0  pose_backend=npy
17:47:30 | INFO    | app.data.isign_dataset | [ISignDataset] split=val  samples=12681  vocab=37535  pose_dim=1728  preload_n=0  pose_backend=npy
17:47:53 | INFO    | app.data.isign_dataset | [ISignDataset] split=test  samples=12681  vocab=37535  pose_dim=1728  preload_n=0  pose_backend=npy


✓ Test dataloader loaded | vocab_size=37535


In [5]:
# Initialize model
# Get pose dimension from config (should be 1728)
pose_input_dim = 1728
use_rgb = not bool(cfg.get('pose_only', False))

model = ISignCSLRModel(
    vocab_size=len(vocab),
    hidden_dim=int(cfg.get('hidden_dim', 256)),
    pose_input_dim=pose_input_dim,
    dropout=float(cfg.get('dropout', 0.2)),
    pretrained_cnn=not bool(cfg.get('no_pretrained', False)),
    pose_fusion_weight=float(cfg.get('pose_fusion_weight', 0.7)),
    attention_heads=int(cfg.get('attention_heads', 4)),
    freeze_rgb_stages=int(cfg.get('freeze_rgb_stages', 4)),
    use_rgb=use_rgb,
).to(device)

checkpoint = load_checkpoint(ckpt_path, model)
model.eval()

print(f"✓ Model loaded | epoch={checkpoint.get('epoch')} | best_wer={checkpoint.get('best_wer'):.4f}")
print(f"  Pose input dim: {pose_input_dim}")
print(f"  Model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

✓ Model loaded | epoch=1 | best_wer=1.0000
  Pose input dim: 1728
  Model parameters: 11.21M


## 3. Pose Visualization Utilities

In [6]:
# MediaPipe Holistic keypoint structure
# Pose: 33 points (x, y, z, visibility)
# Left hand: 21 points (x, y, z)
# Right hand: 21 points (x, y, z)

POSE_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 7),
    (0, 4), (4, 5), (5, 6), (6, 8),
    (9, 10), (11, 12), (11, 13), (13, 15), (15, 17), (15, 21), (15, 19), (17, 19),
    (12, 14), (14, 16), (16, 18), (16, 20), (16, 22), (18, 20), (11, 23), (12, 24),
    (23, 24), (23, 25), (24, 26), (25, 27), (26, 28), (27, 29), (28, 30), (29, 31), (30, 32), (27, 31), (28, 32)
]

HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (5, 9), (9, 10), (10, 11), (11, 12),
    (9, 13), (13, 14), (14, 15), (15, 16),
    (13, 17), (0, 17), (17, 18), (18, 19), (19, 20)
]

def extract_pose_from_flat(flat_pose: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Extract pose, left hand, right hand from flattened keypoint vector.
    Pose: 33*4=132 dims, Left hand: 21*3=63 dims, Right hand: 21*3=63 dims
    Total: 258 dims
    """
    pose = flat_pose[:132].reshape(33, 4)  # x, y, z, visibility
    left_hand = flat_pose[132:195].reshape(21, 3)  # x, y, z
    right_hand = flat_pose[195:258].reshape(21, 3)  # x, y, z
    return pose, left_hand, right_hand

print("✓ Keypoint structure defined")

✓ Keypoint structure defined


In [17]:
import numpy as np
import cv2

def draw_pose_frame(
    pose: np.ndarray,
    left_hand: np.ndarray,
    right_hand: np.ndarray,
    frame_width: int = 640,
    frame_height: int = 480,
    title: str = ""
) -> np.ndarray:

    # -------------------------
    # SAFE FRAME
    # -------------------------
    frame = np.zeros((frame_height, frame_width, 3), dtype=np.uint8)
    frame[:] = (20, 20, 30)

    # -------------------------
    # SANITIZE INPUTS (CRITICAL)
    # -------------------------
    pose = np.nan_to_num(pose, nan=0.0, posinf=0.0, neginf=0.0)
    left_hand = np.nan_to_num(left_hand, nan=0.0, posinf=0.0, neginf=0.0)
    right_hand = np.nan_to_num(right_hand, nan=0.0, posinf=0.0, neginf=0.0)

    # Clamp values to [0,1]
    pose[:, :2] = np.clip(pose[:, :2], 0, 1)
    left_hand[:, :2] = np.clip(left_hand[:, :2], 0, 1)
    right_hand[:, :2] = np.clip(right_hand[:, :2], 0, 1)

    # -------------------------
    # CONVERT TO PIXELS
    # -------------------------
    pose_2d = (pose[:, :2] * np.array([frame_width, frame_height])).astype(np.int32)
    lh_2d = (left_hand[:, :2] * np.array([frame_width, frame_height])).astype(np.int32)
    rh_2d = (right_hand[:, :2] * np.array([frame_width, frame_height])).astype(np.int32)

    visibility = pose[:, 3] if pose.shape[1] > 3 else np.ones(len(pose))

    # -------------------------
    # DRAW POSE CONNECTIONS
    # -------------------------
    for start, end in POSE_CONNECTIONS:
        if start < len(pose_2d) and end < len(pose_2d):
            if visibility[start] > 0.3 and visibility[end] > 0.3:
                pt1 = tuple(pose_2d[start])
                pt2 = tuple(pose_2d[end])

                # extra safety
                if all(0 <= v < 10000 for v in (*pt1, *pt2)):
                    cv2.line(frame, pt1, pt2, (0, 255, 127), 2)

    # -------------------------
    # DRAW POSE POINTS
    # -------------------------
    for i in range(len(pose_2d)):
        if visibility[i] > 0.3:
            pt = tuple(pose_2d[i])
            if 0 <= pt[0] < frame_width and 0 <= pt[1] < frame_height:
                color = (255, 127, 0) if i < 11 else (0, 127, 255)
                cv2.circle(frame, pt, 4, color, -1)

    # -------------------------
    # DRAW HANDS
    # -------------------------
    for hand_2d in [lh_2d, rh_2d]:
        for start, end in HAND_CONNECTIONS:
            if start < len(hand_2d) and end < len(hand_2d):
                pt1 = tuple(hand_2d[start])
                pt2 = tuple(hand_2d[end])

                if (0 <= pt1[0] < frame_width and 0 <= pt1[1] < frame_height and
                    0 <= pt2[0] < frame_width and 0 <= pt2[1] < frame_height):
                    cv2.line(frame, pt1, pt2, (200, 50, 50), 1)

        for pt in hand_2d:
            if 0 <= pt[0] < frame_width and 0 <= pt[1] < frame_height:
                cv2.circle(frame, tuple(pt), 2, (200, 50, 50), -1)

    # -------------------------
    # TITLE
    # -------------------------
    if title:
        cv2.putText(frame, title, (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8,
                    (255, 255, 255), 2)

    return frame

## 4. Generate Motion GIFs from Test Set

In [18]:
def generate_motion_gif(
    pose_sequence: np.ndarray,
    output_path: str,
    gloss: str = "",
    fps: int = 8,
    frame_width: int = 640,
    frame_height: int = 480
) -> None:
    """
    Generate animated GIF from pose sequence.
    
    Args:
        pose_sequence: (num_frames, 258) array
        output_path: path to save GIF
        gloss: text label
        fps: frames per second
        frame_width: canvas width
        frame_height: canvas height
    """
    frames = []
    
    for frame_idx, flat_pose in enumerate(pose_sequence):
        pose, left_hand, right_hand = extract_pose_from_flat(flat_pose)
        
        title = f"{gloss} (frame {frame_idx+1}/{len(pose_sequence)})"
        frame = draw_pose_frame(pose, left_hand, right_hand, frame_width, frame_height, title)
        frames.append(frame[:, :, ::-1])  # BGR to RGB for PIL
    
    # Save as GIF
    imageio.mimsave(output_path, [Image.fromarray(f) for f in frames], fps=fps, loop=0)
    print(f"✓ Saved: {output_path}")

print("✓ GIF generation function defined")

✓ GIF generation function defined


In [9]:
from pathlib import Path
import torch
import numpy as np

# Create output directory
output_dir = Path("motion_gifs")
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {output_dir.resolve()}")

# Collect samples for each unique gloss
gloss_samples = {}  # gloss -> list of samples
max_samples_per_gloss = 3

with torch.no_grad():
    for batch_idx, batch in enumerate(test_loader):

        # Stop early for speed
        if batch_idx >= 200:
            break

        try:
            # -------------------------
            # SAFE BATCH EXTRACTION
            # -------------------------
            pose = batch["pose"].to(device)

            # Handle label length safely
            label_len = int(batch["label_lengths"][0].item())

            # Extract label ids
            ref_ids = batch["labels"][0, :label_len].tolist()

            # Convert ids → tokens
            reference = ids_to_tokens(ref_ids, vocab, blank_id=0)

            # -------------------------
            # SAFE GLOSS EXTRACTION
            # -------------------------
            if reference is not None and len(reference) > 0:
                primary_gloss = str(reference[0])
            else:
                primary_gloss = "UNKNOWN"

            # Initialize dict
            if primary_gloss not in gloss_samples:
                gloss_samples[primary_gloss] = []

            # Limit samples per gloss
            if len(gloss_samples[primary_gloss]) >= max_samples_per_gloss:
                continue

            # -------------------------
            # SAFE POSE CONVERSION
            # -------------------------
            pose_seq = pose[0].detach().cpu().numpy()

            # Force clean numpy array (VERY IMPORTANT)
            pose_seq = np.asarray(pose_seq, dtype=np.float32)
            pose_seq = np.ascontiguousarray(pose_seq)

            # Sentence extraction
            sentence = batch.get("sentences", [""])[0]

            # Store sample
            gloss_samples[primary_gloss].append({
                "idx": batch_idx,
                "pose": pose_seq,
                "gloss": primary_gloss,
                "sentence": sentence
            })

        except Exception as e:
            print(f"⚠️ Skipping batch {batch_idx} due to error: {e}")
            continue


# -------------------------
# FINAL SUMMARY
# -------------------------
total_samples = sum(len(v) for v in gloss_samples.values())

print(f"\n✓ Collected samples for {len(gloss_samples)} unique glosses")
print(f"✓ Total samples collected: {total_samples}")

Output directory: /mnt/d/KATHIR_KALIDASS/CSLR/application/backend/motion_gifs



✓ Collected samples for 114 unique glosses
✓ Total samples collected: 154


In [23]:
# Generate GIFs for collected samples
gif_count = 0
print("pose_data shape:", pose_data.shape)
print("min/max:", pose_data.min(), pose_data.max())
for gloss, samples in sorted(gloss_samples.items()):
    for sample_idx, sample in enumerate(samples):
        gif_name = f"{gloss}_{sample_idx}.gif"
        gif_path = str(output_dir / gif_name)
        
        # Reshape pose from (num_frames, 1728) to (num_frames, 258)
        # 1728 = 258 * (1728/258) = 258 * 6.69... 
        # Actually: 1728 dims / 6 body parts = but let's check structure
        pose_data = sample['pose']
        
        # Verify and reshape correctly
        num_frames = pose_data.shape[0]
        if pose_data.shape[-1] == 1728:
            # reshape properly into chunks of 258
            pose_data = pose_data.reshape(num_frames, -1, 258)

            # take FIRST valid pose chunk
            pose_data = pose_data[:, 0, :]
                
        try:
            generate_motion_gif(
                pose_data,
                gif_path,
                gloss=sample['gloss'],
                fps=8
            )
            gif_count += 1
        except Exception as e:
            print(f"✗ Error generating {gif_name}: {e}")

print(f"\n✓ Generated {gif_count} motion GIFs in {output_dir}")

pose_data shape: (48, 1728)
min/max: -208.87787 544.0556


ValueError: cannot reshape array of size 82944 into shape (48,newaxis,258)

## 5. Display Sample GIFs

In [11]:
from IPython.display import Image as IPImage, display, HTML
import glob

# Get all generated GIFs
gif_files = sorted(glob.glob(str(output_dir / '*.gif')))

print(f"Found {len(gif_files)} GIFs\n")

# Display first 12 GIFs in grid
html_content = """
<div style='display:grid; grid-template-columns: repeat(3, 1fr); gap:20px; padding:20px;'>
"""

for gif_path in gif_files[:12]:
    gif_name = Path(gif_path).stem
    html_content += f"""
    <div style='background:#1a1a2e; padding:10px; border-radius:8px; border:1px solid #444;'>
        <div style='color:#f5f7ff; font-weight:bold; margin-bottom:8px; font-size:14px;'>{gif_name}</div>
        <img src='{gif_path}' width='200' height='150' style='border-radius:4px; display:block;'>
    </div>
    """

html_content += "</div>"

display(HTML(html_content))

Found 0 GIFs



## 6. Generate Comparison: Reference vs Predicted

In [19]:
def generate_comparison_gif(
    reference_pose: np.ndarray,
    predicted_pose: np.ndarray,
    output_path: str,
    gloss: str = "",
    fps: int = 8
) -> None:
    """
    Generate side-by-side comparison GIF of reference vs predicted poses.
    """
    frames = []
    num_frames = min(len(reference_pose), len(predicted_pose))
    
    for frame_idx in range(num_frames):
        # Draw reference (left)
        pose_ref, lh_ref, rh_ref = extract_pose_from_flat(reference_pose[frame_idx][:258])
        frame_ref = draw_pose_frame(pose_ref, lh_ref, rh_ref, 320, 240, "Reference")
        
        # Draw predicted (right)
        pose_pred, lh_pred, rh_pred = extract_pose_from_flat(predicted_pose[frame_idx][:258])
        frame_pred = draw_pose_frame(pose_pred, lh_pred, rh_pred, 320, 240, "Generated")
        
        # Combine side-by-side
        combined = np.hstack([frame_ref, frame_pred])
        frames.append(combined[:, :, ::-1])  # BGR to RGB
    
    imageio.mimsave(output_path, [Image.fromarray(f) for f in frames], fps=fps)
    print(f"✓ Comparison GIF saved: {output_path}")

print("✓ Comparison GIF function defined")

✓ Comparison GIF function defined


In [13]:
# Generate comparison GIFs for first few samples
comp_count = 0

with torch.no_grad():
    for batch_idx, batch in enumerate(test_loader):
        if batch_idx >= 30:  # Create comparisons for first 30
            break
        
        pose = batch['pose'].to(device)
        label_len = int(batch['label_lengths'][0].item())
        ref_ids = batch['labels'][0, :label_len].tolist()
        reference = ids_to_tokens(ref_ids, vocab, blank_id=0)
        primary_gloss = reference[0] if reference else "UNKNOWN"
        
        try:
            # Get model prediction
            log_probs = model(None, pose)
            predicted_gloss = decode_sequences(log_probs, blank_id=0, vocab=vocab)[0]
            
            pose_seq = pose[0].cpu().numpy()
            gif_name = f"comp_{primary_gloss}_vs_{predicted_gloss[0] if predicted_gloss else 'NONE'}_{batch_idx}.gif"
            gif_path = str(output_dir / gif_name)
            
            # For now, create comparison of same sequence to show functionality
            generate_comparison_gif(
                pose_seq,
                pose_seq,  # In practice, this would be model reconstruction
                gif_path,
                gloss=primary_gloss
            )
            comp_count += 1
        except Exception as e:
            pass  # Skip errors

print(f"\n✓ Generated {comp_count} comparison GIFs")


✓ Generated 0 comparison GIFs


## 7. Generate Statistics Report

In [14]:
# Summary statistics
print("\n" + "="*60)
print("MOTION GIF GENERATION SUMMARY")
print("="*60)
print(f"\nModel Configuration:")
print(f"  - Checkpoint: {ckpt_path}")
print(f"  - Best WER: {checkpoint.get('best_wer', 'N/A')}")
print(f"  - Epoch: {checkpoint.get('epoch', 'N/A')}")
print(f"  - Pose Dim: {pose_input_dim}")
print(f"  - Vocab Size: {len(vocab)}")

print(f"\nGenerated Artifacts:")
print(f"  - Output Directory: {output_dir}")
print(f"  - Motion GIFs: {len(gif_files)}")
print(f"  - Unique Glosses: {len(gloss_samples)}")
print(f"  - Total Samples: {sum(len(v) for v in gloss_samples.values())}")

print(f"\nKeypoint Structure:")
print(f"  - Pose: 33 points (x, y, z, visibility)")
print(f"  - Left Hand: 21 points (x, y, z)")
print(f"  - Right Hand: 21 points (x, y, z)")
print(f"  - Total: 258 dimensions per frame")

print(f"\nGIF Settings:")
print(f"  - FPS: 8")
print(f"  - Frame Size: 640x480")
print(f"  - Format: Animated GIF")

print("\n" + "="*60)


MOTION GIF GENERATION SUMMARY

Model Configuration:
  - Checkpoint: checkpoints/isign_pose_only_npy/best.pt
  - Best WER: 1.0
  - Epoch: 1
  - Pose Dim: 1728
  - Vocab Size: 37535

Generated Artifacts:
  - Output Directory: motion_gifs
  - Motion GIFs: 0
  - Unique Glosses: 114
  - Total Samples: 154

Keypoint Structure:
  - Pose: 33 points (x, y, z, visibility)
  - Left Hand: 21 points (x, y, z)
  - Right Hand: 21 points (x, y, z)
  - Total: 258 dimensions per frame

GIF Settings:
  - FPS: 8
  - Frame Size: 640x480
  - Format: Animated GIF



## 8. Export Metadata

In [15]:
# Write metadata JSON
metadata = {
    "generated_at": str(Path.cwd()),
    "checkpoint": str(ckpt_path),
    "best_wer": float(checkpoint.get('best_wer', 0)),
    "epoch": int(checkpoint.get('epoch', 0)),
    "total_gifs": len(gif_files),
    "unique_glosses": len(gloss_samples),
    "samples": {
        gloss: [
            {
                "gloss": s['gloss'],
                "sentence": s['sentence'],
                "frames": s['pose'].shape[0]
            }
            for s in samples
        ]
        for gloss, samples in gloss_samples.items()
    }
}

metadata_path = output_dir / 'metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"✓ Metadata saved to: {metadata_path}")

✓ Metadata saved to: motion_gifs/metadata.json


In [16]:
print("\n✓ Motion GIF visualization complete!")
print(f"All files saved to: {output_dir.absolute()}")


✓ Motion GIF visualization complete!
All files saved to: /mnt/d/KATHIR_KALIDASS/CSLR/application/backend/motion_gifs
